In [123]:
import pandas as pd
import re
import numpy as np
import matplotlib.pyplot as plt

In [124]:
df = pd.read_csv(r"C:\Users\Junayed\nyc-311-resolution-time-analysis\data\raw\311_jan_week1_2025.csv")

In [125]:
df.shape

(90565, 10)

In [126]:
df.columns

Index(['unique_key', 'created_date', 'closed_date', 'agency', 'complaint_type',
       'descriptor', 'borough', 'incident_zip', 'status',
       'open_data_channel_type'],
      dtype='str')

In [127]:
df.dtypes

unique_key                  int64
created_date                  str
closed_date                   str
agency                        str
complaint_type                str
descriptor                    str
borough                       str
incident_zip              float64
status                        str
open_data_channel_type        str
dtype: object

In [128]:
df.head(5)

,unique_key,created_date,closed_date,agency,complaint_type,descriptor,borough,incident_zip,status,open_data_channel_type
0,63572086,2025-01-01T00:51:02.000,2025-03-02T00:51:37.000,DOHMH,Smoking or Vaping,Allowed in Smoke Free Area,BRONX,10475.0,Closed,ONLINE
1,63572092,2025-01-01T00:54:26.000,2025-01-01T01:47:36.000,NYPD,Illegal Parking,Parking Permit Improper Use,QUEENS,11373.0,Closed,MOBILE
2,63572113,2025-01-01T01:14:30.000,2025-01-01T04:38:47.000,NYPD,Blocked Driveway,No Access,BROOKLYN,11208.0,Closed,MOBILE
3,63572115,2025-01-01T01:20:52.000,2025-01-01T02:21:14.000,NYPD,Blocked Driveway,No Access,QUEENS,11420.0,Closed,ONLINE
4,63572131,2025-01-01T01:12:27.000,2025-01-01T04:37:23.000,NYPD,Blocked Driveway,No Access,BROOKLYN,11207.0,Closed,MOBILE


In [129]:
df["unique_key"].isna().sum()

np.int64(0)

# Unique key should be unique, let's test this

In [130]:
print("Unique keys: ")
print(f"Total rows: {len(df)}")
print(f"Unique keys: {df["unique_key"].nunique()}")
print(f"Duplicates: {df["unique_key"].duplicated().sum()}")

Unique keys: 
Total rows: 90565
Unique keys: 90565
Duplicates: 0


### 1. Standardize Column Headers to Title Case

Convert all DataFrame column names to title case (e.g., `unique_key` $\rightarrow$ `Unique_Key`) to maintain a clean, consistent presentation across the dataset.

In [131]:
df.columns = df.columns.str.title()

In [132]:
df.head(5)

,Unique_Key,Created_Date,Closed_Date,Agency,Complaint_Type,Descriptor,Borough,Incident_Zip,Status,Open_Data_Channel_Type
0,63572086,2025-01-01T00:51:02.000,2025-03-02T00:51:37.000,DOHMH,Smoking or Vaping,Allowed in Smoke Free Area,BRONX,10475.0,Closed,ONLINE
1,63572092,2025-01-01T00:54:26.000,2025-01-01T01:47:36.000,NYPD,Illegal Parking,Parking Permit Improper Use,QUEENS,11373.0,Closed,MOBILE
2,63572113,2025-01-01T01:14:30.000,2025-01-01T04:38:47.000,NYPD,Blocked Driveway,No Access,BROOKLYN,11208.0,Closed,MOBILE
3,63572115,2025-01-01T01:20:52.000,2025-01-01T02:21:14.000,NYPD,Blocked Driveway,No Access,QUEENS,11420.0,Closed,ONLINE
4,63572131,2025-01-01T01:12:27.000,2025-01-01T04:37:23.000,NYPD,Blocked Driveway,No Access,BROOKLYN,11207.0,Closed,MOBILE


### Inspect Ticket Status Distribution

Check the frequency distribution of complaint resolution statuses (`df["Status"]`) to understand how many requests are closed, pending, or active within this period.

In [133]:
df["Status"].value_counts()

Status
Closed         90103
In Progress      193
Open             121
Assigned          62
Pending           37
Unspecified       29
Started           20
Name: count, dtype: int64

### Export and Remove Non-Closed Service Requests

Audit, export, and remove all records where `Status` is not `"Closed"`.

**Process:**
* **Export Dropped Data:** Save records with incomplete resolution lifecycles to `Dropped_data.csv` for documentation and auditing.
* **Filter Main Dataset:** Drop these rows by index to ensure subsequent analyses rely exclusively on fully resolved requests with valid `Closed_Date` values.
* **Verification:** Display the count of dropped rows and confirm that only `"Closed"` entries remain.

In [134]:
## keeping the dropping rows
rows_to_drop = df[df["Status"] != "Closed"]
rows_to_drop.to_csv("Dropped_data.csv", index=False)


rows_to_drop = df[df["Status"] != "Closed"].index

total_rows_drop = len(rows_to_drop)
print(f"{total_rows_drop} rows are being dropped.")

df = df.drop(index=rows_to_drop)
df["Status"].value_counts()

462 rows are being dropped.


Status
Closed    90103
Name: count, dtype: int64

### Parse Request Creation Timestamps

Convert the `Created_Date` column from string/object format into standard `datetime64[ns]` objects.

* **Format Handling:** Setting `format="mixed"` accommodates ISO 8601 formatting and potential variations in timestamp representations across rows.
* **Error Handling:** Using `errors='coerce'` converts unparseable strings into `NaT` (Not a Time) rather than interrupting execution.
* **Validation:** Preview the first 10 converted values to verify parsing accuracy.

In [135]:
df["Created_Date"] = pd.to_datetime(df["Created_Date"], format="mixed", errors='coerce')
df["Created_Date"].head(10)

0   2025-01-01 00:51:02
1   2025-01-01 00:54:26
2   2025-01-01 01:14:30
3   2025-01-01 01:20:52
4   2025-01-01 01:12:27
5   2025-01-01 00:07:36
6   2025-01-01 00:19:00
7   2025-01-01 00:11:45
8   2025-01-01 00:14:54
9   2025-01-01 00:17:41
Name: Created_Date, dtype: datetime64[us]

### Parse Request Resolution Timestamps

Convert the `Closed_Date` column from string/object format into `datetime64[ns]` objects to enable downstream time-to-close calculations.

* **Format Handling:** Uses `format="mixed"` to reliably parse standard ISO 8601 timestamps and variations across rows.
* **Error Handling:** Applies `errors='coerce'` to safeguard against corrupt date strings by converting invalid entries into `NaT`.
* **Validation:** Preview the first 10 converted values to verify clean date parsing.

In [136]:
df["Closed_Date"] = pd.to_datetime(df["Closed_Date"], format='mixed', errors='coerce')
df["Closed_Date"].head(10)

0   2025-03-02 00:51:37
1   2025-01-01 01:47:36
2   2025-01-01 04:38:47
3   2025-01-01 02:21:14
4   2025-01-01 04:37:23
5   2025-01-01 01:33:12
6   2025-01-01 00:19:00
7   2025-01-01 01:21:57
8   2025-01-01 01:19:02
9   2025-01-01 00:30:38
Name: Closed_Date, dtype: datetime64[us]

### Check for Missing Resolution Timestamps

Verify data completeness by calculating the total number of missing (`NaT` / `NaN`) values in `Closed_Date` following the datetime conversion. 

* Even within records marked as `"Closed"`, missing timestamps (e.g., records closed administratively without an explicit resolution timestamp or unparseable date strings) cannot be used in duration calculations.
* This count determines whether further row pruning is required before calculating turnaround times.

In [137]:
df["Closed_Date"].isna().sum()

np.int64(230)

### Impute Missing Resolution Dates

Handle the 230 records flagged as `"Closed"` that are missing an explicit `Closed_Date` timestamp by applying an end-of-week (EOW) business assumption:

* **Audit Annotation:** Update the `Descriptor` field to `"No closed date, I assumed they were closed at EOW."` to ensure transparency and traceability for downstream consumers.
* **Timestamp Imputation:** Impute missing `Closed_Date` entries with the end of the analysis period (`2025-01-07 23:59:59`).
* **Validation:** Re-evaluate `isna().sum()` on `Closed_Date` to verify zero null timestamps remain.

In [138]:
df.loc[df["Closed_Date"].isna(), "Descriptor"] = "No closed date, I assumed they were closed at EOW."

In [139]:
df.loc[df["Closed_Date"].isna(), "Closed_Date"] = pd.to_datetime("2025-01-07 23:59:59")
df["Closed_Date"].isna().sum()

np.int64(0)

### Compute Ticket Resolution Time in Days

Engineer a new metric, `Resolution_Days`, to quantify how long each service request took to resolve:

* **Calculation:** Subtract complaint creation time (`Created_Date`) from resolution time (`Closed_Date`), extracting the integer component via `.dt.days`.
* **Metric Definition:** Captures the full elapsed calendar days required by agencies to address and close each ticket.
* **Preview:** Display the first 10 computed values to confirm proper feature derivation.

In [140]:
df["Resolution_Days"] = (df["Closed_Date"] - df["Created_Date"]).dt.days
df["Resolution_Days"].head(10)

0    60
1     0
2     0
3     0
4     0
5     0
6     0
7     0
8     0
9     0
Name: Resolution_Days, dtype: int64

### Analyze Same-Day Resolution Volume

Quantify how many service requests were resolved within 24 hours of submission (`Resolution_Days == 0`).

* **Metric Finding:** 56,320 tickets were closed on the same calendar day as intake.
* **Operational Significance:** A large cluster of same-day closures indicates high throughput for immediate or rapid-dispatch complaint types (such as parking enforcement or immediate field inspections).

In [141]:
print(f"Same-day reolutions(0 days): {(df["Resolution_Days"] == 0).sum()}")

Same-day reolutions(0 days): 56320


### Analyze Resolution Duration Distribution (Day-by-Day)

Segment closed service requests across daily resolution brackets (from 1 to 7 days, and extended turnaround exceeding one week):

* **Short-Term Turnaround:** The vast majority of multi-day resolutions are concluded rapidly within 1 to 2 days (12,716 and 5,392 cases, respectively).
* **Diminishing Volume:** Ticket volume drops off steadily past day 3, indicating efficient resolution for typical standard cases.
* **Extended Backlog:** A substantial cohort of 9,780 complaints took longer than 7 days to close, highlighting complex investigations, agency handoffs, or administrative backlogs.

In [142]:
print(
    f"Took 1 day: {((df['Resolution_Days'] > 0) & (df['Resolution_Days'] <= 1)).sum()}"
)
print(
    f"Took 2 day: {((df['Resolution_Days'] > 1) & (df['Resolution_Days'] <= 2)).sum()}"
)
print(
    f"Took 3 day: {((df['Resolution_Days'] > 2) & (df['Resolution_Days'] <= 3)).sum()}"
)
print(
    f"Took 4 day: {((df['Resolution_Days'] > 3) & (df['Resolution_Days'] <= 4)).sum()}"
)
print(
    f"Took 5 day: {((df['Resolution_Days'] > 4) & (df['Resolution_Days'] <= 5)).sum()}"
)
print(
    f"Took 6 day: {((df['Resolution_Days'] > 5) & (df['Resolution_Days'] <= 6)).sum()}"
)
print(
    f"Took 7 day: {((df['Resolution_Days'] > 6) & (df['Resolution_Days'] <= 7)).sum()}"
)
print(
    f"Took more than 7 days: {(df["Resolution_Days"] > 7).sum()}"
)

Took 1 day: 12716
Took 2 day: 5392
Took 3 day: 2602
Took 4 day: 1133
Took 5 day: 749
Took 6 day: 764
Took 7 day: 645
Took more than 7 days: 9780


### Reconcile Dataset Counts & Identify Temporal Anomalies

Perform a cross-check between total active records and the daily resolution breakdown:

* **Row Count Discrepancy:** The total closed dataset contains **90,103** rows, whereas the non-negative duration buckets ($0$ to $>7$ days) accounted for **90,101** records, leaving 2 untracked rows.
* **Root Cause Identification:** Evaluating `Resolution_Days < 0` confirms exactly **2 anomalous records** where resolution timestamps predate the creation timestamps.
* **Operational Implication:** Negative turnaround times violate causal chronology (tickets cannot be completed before being logged), indicating manual back-dating, legacy dispatch linking, or clerical data entry errors.

In [143]:
len(df)

90103

In [144]:
print(
    f"?: {(df["Resolution_Days"] <0).sum()}"
)

?: 2


### Inspect Records with Negative Turnaround Times

Isolate the key attributes (`Unique_Key`, `Created_Date`, `Closed_Date`, `Agency`, `Descriptor`, `Complaint_Type`, `Borough`, `Incident_Zip`) for the two records exhibiting `Resolution_Days < 0`:

* **Key Findings:**
  * Both records are Department of Transportation (`DOT`) complaints concerning `Sidewalk Condition` (`Blocked - Construction`) in the `BRONX` (ZIP `10452`).
  * Both tickets were opened on the evening of **January 1, 2025** (at 19:07 and 20:19), but their resolution timestamp is recorded as **December 4, 2024, at 09:33:00** (approximately 28–29 days prior to ticket submission).
* **Diagnosis:** Identical descriptors, location, and resolution timestamps indicate duplicate complaints retroactively linked or batched against an earlier, pre-existing DOT inspection/closure event from the previous month.
* **Cleaning Recommendation:** These records represent temporal contradictions and should be excluded from duration calculations to avoid skewing summary statistics.

In [145]:
df.loc[df["Resolution_Days"] < 0, ["Unique_Key","Created_Date","Closed_Date","Agency","Descriptor","Complaint_Type","Borough","Incident_Zip"]]

,Unique_Key,Created_Date,Closed_Date,Agency,Descriptor,Complaint_Type,Borough,Incident_Zip
6736,63591237,2025-01-01 19:07:52,2024-12-04 09:33:00,DOT,Blocked - Construction,Sidewalk Condition,BRONX,10452.0
7750,63592286,2025-01-01 20:19:16,2024-12-04 09:33:00,DOT,Blocked - Construction,Sidewalk Condition,BRONX,10452.0


### Remove Negative Turnaround Anomalies

Filter out the two records with `Resolution_Days < 0` where `Closed_Date` precedes `Created_Date`.

* **Objective:** Ensure temporal validity across the dataset by retaining only tickets where resolution occurred at or after complaint creation (`Resolution_Days >= 0`).
* **Impact:** Eliminates data contamination in subsequent descriptive statistics (mean, median) and duration distribution analyses.

In [146]:
df = df.drop(index=[6736, 7750])

In [147]:
df["Complaint_Type"].value_counts()

Complaint_Type
Noise - Residential               30227
HEAT/HOT WATER                    14990
Illegal Parking                    9745
Blocked Driveway                   3297
UNSANITARY CONDITION               2215
                                  ...  
Ferry Complaint                       1
Building Condition                    1
Scaffold Safety                       1
Institution Disposal Complaint        1
Highway Sign - Damaged                1
Name: count, Length: 141, dtype: int64

### Standardize Complaint Type Text Formatting

Normalize string casing across the `Complaint_Type` column by casting values to string and converting them to title case (`.str.title()`):

* **Consistency:** Eliminates casing discrepancies (such as mixing UPPERCASE agency acronyms, lowercase entries, or mixed casing across intake channels) that could cause identical issues to be counted as distinct categories.
* **Readability:** Provides clean, uniform category labels for subsequent group-by summaries and visualizations.

In [148]:
df["Complaint_Type"] = df["Complaint_Type"].astype(str).str.title()

### Standardize Borough Names & Audit Null Values

Normalize geographic designations in the `Borough` column and verify data completeness:

* **Format Harmonization:** Cast the column to string and apply `.str.title()` to convert values into standard title casing (e.g., `"BRONX"` $\rightarrow$ `"Bronx"`, `"STATEN ISLAND"` $\rightarrow$ `"Staten Island"`).
* **Missing Value Audit:** Compute `.isna().sum()` to confirm whether any records lack borough assignments.

In [149]:
df["Borough"] = df["Borough"].astype(str).str.title()
df["Borough"].isna().sum()

np.int64(0)

### Analyze Service Request Distribution by Borough

Examine the geographic volume of closed service requests across New York City boroughs:

* **Top Reporter:** The **Bronx** accounts for the highest share of requests by a wide margin with **39,572** cases, followed by **Brooklyn** (**20,159**).
* **Mid-to-Lower Volume:** **Queens** (**14,906**) and **Manhattan** (**13,252**) exhibit comparable reporting volumes, while **Staten Island** logs **2,177** records.
* **Data Hygiene Check:** Only **35** entries are recorded as `"Unspecified"`, indicating strong geographic completeness overall ($< 0.04\%$ unassigned).

In [150]:
df["Borough"].value_counts()

Borough
Bronx            39572
Brooklyn         20159
Queens           14906
Manhattan        13252
Staten Island     2177
Unspecified         35
Name: count, dtype: int64

### Filter Out Unspecified Borough Records

Prune records where `Borough` is recorded as `"Unspecified"` to ensure high geographic integrity for spatial and regional reporting:

* **Volume Dropped:** Removes **35** unassigned entries, reducing the dataset from **90,101** to **90,066** rows ($< 0.04\%$ data reduction).
* **Rationale:** Spatial aggregation and comparative borough analyses require confirmed administrative boundaries; keeping unassigned records would distort per-borough rate and volume calculations.
* **Verification:** Confirm the post-filter row count reflects exactly 90,066 valid records.

In [151]:
df = df[df["Borough"] != "Unspecified"]
print(f"After dropping Unspecified borough: {len(df)} rows")

After dropping Unspecified borough: 90066 rows


### Audit Missing Incident ZIP Codes & Geographic Granularity

Assess completeness in the `Incident_Zip` column and evaluate whether missing values can be imputed from `Borough`:

* **Missing Values Audit:** Identified **497** closed records lacking postal ZIP code entries.
* **Granularity Check:** Inspecting postal codes within a single borough (e.g., `Borough == "Bronx"`) confirms that each NYC borough encompasses dozens of distinct ZIP codes.
* **Imputation Assessment:** Because the relationship between borough and ZIP code is one-to-many rather than one-to-one, missing `Incident_Zip` values cannot be reliably imputed using borough data alone without introducing geographic distortion.

In [152]:
df["Incident_Zip"].isna().sum()

np.int64(465)

In [153]:
df.loc[df["Borough"] == "Bronx", ["Borough","Incident_Zip"]]

,Borough,Incident_Zip
0,Bronx,10475.0
11,Bronx,10469.0
29,Bronx,10467.0
33,Bronx,10474.0
43,Bronx,10451.0
...,...,...
90556,Bronx,10466.0
90557,Bronx,10472.0
90558,Bronx,NaN
90559,Bronx,10461.0


### Filter Out Missing Postal Codes (`Incident_Zip`)

Purge records missing postal codes to establish complete geographic attribution across all retained service requests:

* **Filter Criterion:** Retain only rows where `Incident_Zip` is non-null using `.notna()`.
* **Data Reduction:** Drops exactly **465** unassigned ZIP records, bringing total active rows from **90,066** down to **89,601** ($0.52\%$ dropped).
* **Downstream Benefit:** Prevents null handling issues during neighborhood-level aggregations, ZIP-code choropleth mapping, and spatial trend analysis.

In [154]:
df = df[df["Incident_Zip"].notna()]
print(f"After dropping NaN zip codes : {len(df)} rows")

After dropping NaN zip codes : 89601 rows


### Analyze Intake Channel Distribution (`Open_Data_Channel_Type`)

Examine how citizens submit service requests across reporting channels:

* **Primary Channels:** Mobile applications lead intake with **36,691** submissions, followed closely by online web portals with **30,260** records.
* **Traditional vs. Digital:** Digital self-service channels (`MOBILE` + `ONLINE`) account for the vast majority of intake, substantially outpacing traditional phone-in requests (`PHONE`: **18,703**).
* **Data Quality Note:** **4,447** records are logged under `UNKNOWN` intake sources, representing automated ingest pipelines, API bulk imports, or legacy dispatch entries.

In [155]:
df["Open_Data_Channel_Type"].value_counts()

Open_Data_Channel_Type
MOBILE     36511
ONLINE     30163
PHONE      18580
UNKNOWN     4347
Name: count, dtype: int64

### Standardize Intake Channel Casing

Normalize the `Open_Data_Channel_Type` column by casting values to string and converting all labels to title case (`.str.title()`):

* **Consistency:** Aligns intake channel values (e.g., `"MOBILE"` $\rightarrow$ `"Mobile"`, `"ONLINE"` $\rightarrow$ `"Online"`) with the title-cased formatting used throughout the dataset.
* **Presentation:** Ensures clean, professional categorical labels for summary tables, grouping operations, and downstream plots.

In [156]:
df["Open_Data_Channel_Type"] = df["Open_Data_Channel_Type"].astype(str).str.title()
df["Open_Data_Channel_Type"].value_counts()

Open_Data_Channel_Type
Mobile     36511
Online     30163
Phone      18580
Unknown     4347
Name: count, dtype: int64